# AgriTrust — Crop Vision Model Training
Downloads the Kaggle agriculture-crops-dataset, preprocesses it, and trains a YOLOv8 classification model.

**Output:** `ml_weights/crop_classifier.pt`

### Before running:
Set your Kaggle credentials in one of two ways:
```bash
# Option A — environment variables
export KAGGLE_USERNAME=your_username
export KAGGLE_KEY=your_api_key

# Option B — place kaggle.json at ~/.kaggle/kaggle.json
```

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'data/training'))
print('Repo root:', REPO_ROOT)

In [ ]:
# Install dependencies if needed
# !pip install kagglehub ultralytics pyyaml

## 1. Download Dataset from Kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download('osamajalilhassan/agriculture-crops-dataset')
print('Dataset downloaded to:', path)

## 2. Inspect Dataset Structure

In [ ]:
from pathlib import Path

dataset_path = Path(path)
print('Top-level contents:')
for item in sorted(dataset_path.iterdir()):
    n_files = sum(1 for _ in item.rglob('*') if _.is_file()) if item.is_dir() else 1
    print(f'  {item.name}/  ({n_files} files)')

## 3. Preprocess — Merge Local + Kaggle Datasets

In [ ]:
# This merges:
#   data/training/data/local_crops/   (30 classes — already copied from your Desktop)
#   data/training/data/kaggle_crops/  (Kaggle dataset — optional, skip if not downloaded)
import subprocess, sys
result = subprocess.run(
    [sys.executable, os.path.join(REPO_ROOT, 'data/training/scripts/merge_datasets.py')],
    capture_output=False
)
print('Exit code:', result.returncode)

## 4. Verify Preprocessed Data

In [ ]:
import json

DATASET_ROOT = Path(REPO_ROOT) / 'data/training/data/merged_crops'
CLASS_MAP    = DATASET_ROOT / 'class_map.json'

with open(CLASS_MAP) as f:
    class_map = json.load(f)

print(f'Classes ({len(class_map)}):')
for idx, name in class_map.items():
    train_count = len(list((DATASET_ROOT / 'images' / 'train' / name).glob('*'))) if (DATASET_ROOT / 'images' / 'train' / name).exists() else 0
    val_count   = len(list((DATASET_ROOT / 'images' / 'val'   / name).glob('*'))) if (DATASET_ROOT / 'images' / 'val'   / name).exists() else 0
    print(f'  [{idx}] {name}: {train_count} train / {val_count} val')

## 5. Sample Images Preview

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

train_dir = DATASET_ROOT / 'images' / 'train'
classes   = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
sample_classes = classes[:min(9, len(classes))]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, cls in zip(axes.flatten(), sample_classes):
    imgs = list((train_dir / cls).glob('*'))
    if imgs:
        img = mpimg.imread(random.choice(imgs))
        ax.imshow(img)
        ax.set_title(cls, fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Train YOLOv8 Classification Model
Change `device` to `'cuda'` if you have a GPU.

In [ ]:
from ultralytics import YOLO
import shutil
from datetime import datetime

WEIGHTS_DIR   = Path(REPO_ROOT) / 'ml_weights'
WEIGHTS_DIR.mkdir(exist_ok=True)

EPOCHS  = 50    # increase to 100 for better accuracy
IMGSZ   = 224
BATCH   = 32
DEVICE  = 'cpu' # change to 'cuda' if GPU available

model = YOLO('yolov8n-cls.pt')  # nano classification model

run_name = f"crop_cls_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

results = model.train(
    data=str(DATASET_ROOT.resolve()),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(WEIGHTS_DIR / 'runs'),
    name=run_name,
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
)

# Copy best weights to canonical path
best_pt = WEIGHTS_DIR / 'runs' / run_name / 'weights' / 'best.pt'
if best_pt.exists():
    shutil.copy2(best_pt, WEIGHTS_DIR / 'crop_classifier.pt')
    print(f'Best weights saved → ml_weights/crop_classifier.pt')
else:
    print(f'WARNING: best.pt not found at {best_pt}')

## 7. Validate

In [ ]:
trained = YOLO(str(WEIGHTS_DIR / 'crop_classifier.pt'))
metrics = trained.val(data=str(DATASET_ROOT.resolve()))
print(f'Top-1 Accuracy: {metrics.top1:.4f}')
print(f'Top-5 Accuracy: {metrics.top5:.4f}')

## 8. Test on a Single Image

In [ ]:
# Pick any image from the val set
val_dir = DATASET_ROOT / 'images' / 'val'
test_img = next(val_dir.rglob('*.jpg'), None) or next(val_dir.rglob('*.png'), None)

if test_img:
    result = trained(str(test_img))[0]
    top_class = result.probs.top1
    top_conf  = result.probs.top1conf.item()
    print(f'Predicted: {class_map[str(top_class)]}  ({top_conf*100:.1f}% confidence)')
    
    img = mpimg.imread(test_img)
    plt.imshow(img)
    plt.title(f'Predicted: {class_map[str(top_class)]} ({top_conf*100:.1f}%)')
    plt.axis('off')
    plt.show()
else:
    print('No test image found')